# SAR preprocess experiments and `.cos` files to HDF5
**This notebook is most likely deprecated compared to the code in `scripts/TSX_dataset_creation.py`.**

However, it is the good place to experiment with:
1. Loading all CoSAR-format (`.cos`) images from `data/TSX_cos_files`.
2. Applying the SAR preprocessing pipeline (symmetrization, scatterer preservation, log‑normalization, patch extraction).
3. Spliting into train/val/test.
4. Saving each split as an HDF5 file for fast loading during training.
5. Visualizing intermediate results for debugging.

In [ ]:
# Imports
from pathlib import Path
import numpy as np
import torch
import h5py
import matplotlib.pyplot as plt
import os
import random
import hashlib
import sys
from torch.utils.data import Dataset, random_split
from tqdm.notebook import tqdm
from datetime import datetime

# Add parent directory to path to import from src
sys.path.append("..")

# Import original SAR utilities
import src.utils.MERLIN_sar_utils as MERLIN_sar_utils
from src.utils.sar_utils import print_sar_statistics
# from src.utils.MERLIN_sar_utils import (
#     cos2mat,
#     normalize_sar,
#     normalize_sar_as_network_input,
#     denormalize_sar,
#     symetrisation_patch_test,
# )
# from src.utils.sar_utils import print_sar_statistics

# Quick ANSI color code shortcuts
r = "\033[31m"  # Red
y = "\033[33m"  # Yellow
g = "\033[32m"  # Green
b = "\033[34m"  # Blue
e = "\033[0m"  # Reset

# Set random seeds for reproducibility
general_seed = 42
np.random.seed(general_seed)
torch.manual_seed(general_seed)
random.seed(general_seed)

In [ ]:
# Configuration
data_dir = Path("../data/TSX_cos_files")  # Adjust path as needed
output_dir = Path("../data/processed_hdf5")
output_dir.mkdir(exist_ok=True, parents=True)

patch_size = 256
seed = 42
train_frac = 0.8
val_frac = 0.1  # test_frac will be the remaining 0.1

# Verify the data directory exists
if not data_dir.exists():
    print(f"Warning: Data directory {b}{data_dir}{e} not found. Creating it.")
    data_dir.mkdir(exist_ok=True, parents=True)

print(f"Data directory: {b}{data_dir}{e}")
print(f"Output directory: {b}{output_dir}{e}")

In [ ]:
print(10 * np.log10(30276))

## 1. Test Processing Pipeline with a Single Image

First, implement the necessary functions to load and process SAR images using the original utilities.
Then, test our pipeline with a single image before processing the whole dataset.

In [ ]:
def write_hdf5(patches, path):
    """Write dataset patches to HDF5 at path.

    Args:
        real_patches: Real part patches
        imag_patches: Imaginary part patches
        intensity_patches: Intensity patches
        path: Path to save HDF5 file
    """
    n_patches = len(patches)
    print(f"Writing {n_patches} patches to {path}...")

    try:
        with h5py.File(path, "w") as f:
            f.create_dataset("patches", data=patches, dtype="float32")

            # Add metadata
            f.attrs["num_patches"] = n_patches
            f.attrs["patch_size"] = patch_size
            f.attrs["creation_date"] = str(datetime.now())

        print(f"Successfully wrote {n_patches} patches to {path}")
        return True
    except Exception as e:
        print(f"Error writing HDF5 file: {e}")
        return False

In [ ]:
def extract_patches(data, patch_size, stride=None):
    """Extract patches of size patch_size from the input data.

    Args:
        data: Input data array (2D or 3D)
        patch_size: Size of patches to extract (patch_size x patch_size)
        stride: Stride for extraction (default: patch_size for no overlap)

    Returns:
        List of extracted patches
    """
    if stride is None:
        stride = patch_size  # Default: no overlap

    patches = []

    # Check if we're dealing with 3D data (real+imag channels)
    if len(data.shape) == 3:
        h, w, _ = data.shape
        for i in range(0, h - patch_size + 1, stride):
            for j in range(0, w - patch_size + 1, stride):
                patch = data[i : i + patch_size, j : j + patch_size, :]
                if patch.shape[:2] == (patch_size, patch_size):
                    patches.append(patch)
        return np.array(patches)
    else:  # 2D data (intensity only)
        h, w = data.shape
        for i in range(0, h - patch_size + 1, stride):
            for j in range(0, w - patch_size + 1, stride):
                patch = data[i : i + patch_size, j : j + patch_size]
                if patch.shape == (patch_size, patch_size):
                    patches.append(patch)
        return np.array(patches)

In [ ]:
def preserve_scatterers(real2, imag2, threshold_db=9):
    """Preserve point-like scatterers in SAR image above a certain threshold.

    Args:
        real2: Squared real part of SAR image
        imag2: Squared imaginary part of SAR image
        threshold_db: Threshold in dB for scatterer preservation (default: 9dB)

    Returns:
        Tuple of (restacked_data, scatterer_mask)
    """
    # Convert intensity to dB (10*log10(intensity)), add a small epsilon to avoid log(0)
    intensity = real2 + imag2
    intensity_db = 10 * np.log10(intensity + 1e-10)

    # Create mask for scatterers above threshold
    scatterer_mask = intensity_db > threshold_db

    # Create copies of the input arrays to avoid modifying originals
    real2_proc = real2.copy()
    imag2_proc = imag2.copy()

    # For pixels above threshold, assign the same value to both real and imaginary parts
    # Value = sqrt(intensity/2), which gives half the power to each component
    scatterer_value = np.sqrt(intensity[scatterer_mask] / 2)
    real2_proc[scatterer_mask] = scatterer_value
    imag2_proc[scatterer_mask] = scatterer_value

    return np.stack((real2_proc, imag2_proc), axis=2), scatterer_mask

In [ ]:
# New processing pipeline order:
# 1. Load real and imag
# 2. patchification
# 3. symmetrization
# 4. square real and imag
# 5. scatterer preservation
# 6. normalization


def preprocess_sar_image(filepath):
    """Full preprocessing pipeline for a single SAR image using original functions.

    Args:
        filepath: Path to the SAR image file

    Returns:
        Dictionary with original_data, patches at different processing stages
    """
    print(f"Processing {filepath}...")

    # 1. Load SAR data
    print(f"    1. Loading SAR data from {filepath}...")
    sar_data = MERLIN_sar_utils.cos2mat(str(filepath))
    if sar_data is None:
        raise ValueError(f"Failed to load {filepath}")
    # Store original data
    original_data = sar_data.copy()

    # 2. Extract patches with NO OVERLAP (stride=patch_size//2 for 50% overlap)
    print(
        f"    2. Extracting patches of size {patch_size}x{patch_size} with no overlap..."
    )
    original_patches = extract_patches(original_data, patch_size, stride=patch_size)
    print(
        f"          Extracted {len(original_patches)} patches of size {patch_size}x{patch_size}"
    )

    # 3. Apply symmetrization to each patch
    print("    3. Applying symmetrization to each patch...")
    symmetrized_patches = []
    for patch in original_patches:
        # Reshape to match MERLIN's expected format: [h, w, 2] -> real_ and imag_part [1, h, w, 1]
        real_part = patch[:, :, 0]
        imag_part = patch[:, :, 1]
        real_part_reshaped = real_part.reshape(1, *real_part.shape, 1)
        imag_part_reshaped = imag_part.reshape(1, *imag_part.shape, 1)
        real_sym, imag_sym = MERLIN_sar_utils.symetrisation_patch_test(
            real_part_reshaped, imag_part_reshaped
        )
        # Reshape back to [h, w, 2] format
        real_sym = real_sym[0, :, :, 0]
        imag_sym = imag_sym[0, :, :, 0]
        symmetrized_patch = np.stack((real_sym, imag_sym), axis=2)
        symmetrized_patches.append(symmetrized_patch)

    symmetrized_patches = np.array(symmetrized_patches)

    # 4. Square the real and imaginary parts (step included in visualization)
    print("    4. Squaring the real and imaginary parts of the patches...")
    symmetrized_patches_squared = np.square(symmetrized_patches)
    print(
        f"          Squared the real and imaginary parts of the patches. Shape: {symmetrized_patches_squared.shape}"
    )

    # 5. Preserve scatterers in each patch
    print("    5. Preserving point-like scatterers in each patch...")
    preserved_patches_squared = []
    scatterer_masks = []
    for patch in symmetrized_patches_squared:
        preserved_patch_squared, scatterer_mask = preserve_scatterers(
            patch[:, :, 0], patch[:, :, 1], threshold_db=35
        )
        preserved_patches_squared.append(preserved_patch_squared)
        scatterer_masks.append(scatterer_mask)

    preserved_patches_squared = np.array(preserved_patches_squared)
    scatterer_masks = np.array(scatterer_masks)

    # Return the results at different stages of the pipeline
    return {
        "original_data": original_data,
        "original_patches": original_patches,
        "symmetrized_patches": symmetrized_patches,
        "scatterer_masks": scatterer_masks,
        "patches": preserved_patches_squared,  # Final processed patches
    }

In [ ]:
# List .cos files in the data directory
cos_files = list(data_dir.glob("*.cos"))
print(f"Found {len(cos_files)} .cos files in {data_dir}:")
for file in cos_files:
    print(f"  - {file.name}")
# Use the first real file
test_file_path = cos_files[2]

# Images: Genoa (2009, E009-045), Cologne (2020, E006-493), Hamburg (2018, E009-901)

# Process the test file
results = preprocess_sar_image(test_file_path)

In [ ]:
def visualize_1_patch_processing_and_statistics(
    original_patch,
    symmetrized_patch,
    preserved_patch,
    scatterer_mask=None,
):
    """Visualization and statistics of SAR preprocessing steps.

    Args:
        original_patch: Original patch [h, w, 2]
        symmetrized_patch: Symmetrized patch [h, w, 2]
        preserved_patch: Final processed patch with scatterer preservation (/!\\ already squared) [h, w, 2]
        scatterer_mask: Optional mask showing preserved scatterers [h, w]
    """
    print_sar_statistics("original_patch", original_patch)
    print_sar_statistics("symmetrized_patch", symmetrized_patch)
    print_sar_statistics("preserved_patch (squared)", preserved_patch)

    # Normalization (and squaring when necessary)
    orig_patch_norm = MERLIN_sar_utils.normalize_sar(np.square(original_patch))
    sym_patch_norm = MERLIN_sar_utils.normalize_sar(np.square(symmetrized_patch))
    pres_patch_norm = MERLIN_sar_utils.normalize_sar(preserved_patch)  # already squared

    # Create figure with subplots
    ncols = 3
    fig, axes = plt.subplots(3, ncols, figsize=(5 * ncols, 15))
    fig.suptitle(
        "Even original and symmetrized are squared (before normalization).",
        fontsize=16,
    )

    # Original patch
    im11 = axes[0, 0].imshow(original_patch[:, :, 0] ** 2, cmap="gray")
    axes[0, 0].set_title("Original patch real")
    fig.colorbar(im11, ax=axes[0, 0], shrink=0.7)
    im12 = axes[0, 1].imshow(orig_patch_norm[:, :, 0], cmap="gray")
    axes[0, 1].set_title("Original patch normalized real")
    fig.colorbar(im12, ax=axes[0, 1], shrink=0.7)
    orig_patch_norm_intensity = (
        orig_patch_norm[:, :, 0] ** 2 + orig_patch_norm[:, :, 1] ** 2
    )
    im13 = axes[0, 2].imshow(orig_patch_norm_intensity, cmap="gray")
    axes[0, 2].set_title("Original patch normalized intensity")
    fig.colorbar(im13, ax=axes[0, 2], shrink=0.7)

    # Symmetrized patch
    im21 = axes[1, 0].imshow(symmetrized_patch[:, :, 0] ** 2, cmap="gray")
    axes[1, 0].set_title("Symmetrized patch real")
    fig.colorbar(im21, ax=axes[1, 0], shrink=0.7)
    im22 = axes[1, 1].imshow(sym_patch_norm[:, :, 0], cmap="gray")
    axes[1, 1].set_title("Symmetrized patch normalized real")
    fig.colorbar(im22, ax=axes[1, 1], shrink=0.7)
    sym_patch_norm_intensity = (
        sym_patch_norm[:, :, 0] ** 2 + sym_patch_norm[:, :, 1] ** 2
    )
    im23 = axes[1, 2].imshow(sym_patch_norm_intensity, cmap="gray")
    axes[1, 2].set_title("Symmetrized patch normalized intensity")
    fig.colorbar(im23, ax=axes[1, 2], shrink=0.7)

    # Final preserved patch
    im7 = axes[2, 0].imshow(preserved_patch[:, :, 0], cmap="gray")
    axes[2, 0].set_title("Preserved patch real")
    fig.colorbar(im7, ax=axes[2, 0], shrink=0.7)
    im8 = axes[2, 1].imshow(pres_patch_norm[:, :, 0], cmap="gray")
    axes[2, 1].set_title("Preserved patch normalized real")
    fig.colorbar(im8, ax=axes[2, 1], shrink=0.7)
    pres_patch_norm_intensity = pres_patch_norm[:, :, 0] + pres_patch_norm[:, :, 1]
    im9 = axes[2, 2].imshow(pres_patch_norm_intensity, cmap="gray")
    axes[2, 2].set_title("Preserved patch normalized intensity")
    fig.colorbar(im9, ax=axes[2, 2], shrink=0.7)

    plt.tight_layout()
    plt.show()

    # If we have a scatterer mask, show it as a second small figure
    if scatterer_mask is not None:
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(scatterer_mask, cmap="gray")
        ax.set_title("Strong scatterer mask")
        fig.colorbar(ax.imshow(scatterer_mask, cmap="gray"), ax=ax, shrink=0.7)
        plt.tight_layout()
        plt.show()

        _, nb_scatterer_preserved = np.unique(scatterer_mask, return_counts=True)
        print(
            f"Strong scatterers preserved = {nb_scatterer_preserved[0]} (or {nb_scatterer_preserved[0] / nb_scatterer_preserved[1] * 100:.6f}%)."
        )


In [ ]:
def plot_intensity_histogram(data, title="Log-Intensity Histogram", is_squared=False):
    """Plot histogram of log-intensity values and print key statistics.

    Args:
        data: SAR data in [h, w, 2] format (single image) or [n, h, w, 2] format (batch of patches)
        title: Title for the plot
        is_squared: If True, assumes the data is already squared (e.g., preserved_patches)
    """
    # Check if we're dealing with a batch of patches or a single image
    is_batch = len(data.shape) == 4  # [n, h, w, 2] format

    # Initialize an empty list to collect flattened intensity values
    intensity_values = []

    if is_batch:
        # print(f"Processing batch of {len(data)} patches...")
        # Process each patch separately and collect flattened intensity values
        for patch in data:
            if is_squared:
                # If data is already squared (like preserved_patches)
                intensity = patch[:, :, 0] + patch[:, :, 1]
            else:
                # Calculate intensity for non-squared data
                intensity = patch[:, :, 0] ** 2 + patch[:, :, 1] ** 2
            # Flatten and append to our list
            intensity_values.append(intensity.flatten())

        # Concatenate all flattened arrays into a single 1D array
        intensity_all = np.concatenate(intensity_values)
        # print(f"Final intensity array shape: {intensity_all.shape} (should be 1D)")
    else:
        # Single image processing
        if is_squared:
            intensity = data[:, :, 0] + data[:, :, 1]
        else:
            intensity = data[:, :, 0] ** 2 + data[:, :, 1] ** 2
        intensity_all = intensity.flatten()

    # Convert to dB (log scale)
    epsilon = 1e-10  # Small value to avoid log(0)
    log_intensity = 10 * np.log10(intensity_all + epsilon)

    # Calculate statistics
    mean_val = np.mean(log_intensity)
    median_val = np.median(log_intensity)
    p1 = np.percentile(log_intensity, 1)
    p5 = np.percentile(log_intensity, 5)
    p10 = np.percentile(log_intensity, 10)
    p90 = np.percentile(log_intensity, 90)
    p95 = np.percentile(log_intensity, 95)
    p99 = np.percentile(log_intensity, 99)

    # Print statistics
    print(f"Log-Intensity Statistics for {title} (dB):")
    print(f"  Mean: {mean_val:.2f} dB")
    print(f"  Median: {median_val:.2f} dB")
    print(f"  1st percentile: {p1:.2f} dB")
    print(f"  5th percentile: {p5:.2f} dB")
    print(f"  10th percentile: {p10:.2f} dB")
    print(f"  90th percentile: {p90:.2f} dB")
    print(f"  95th percentile: {p95:.2f} dB")
    print(f"  99th percentile: {p99:.2f} dB")

    # Plot histogram
    plt.figure(figsize=(10, 6))
    hist, bins, _ = plt.hist(log_intensity, bins=100, color="blue", alpha=0.7)

    # Add vertical lines for key statistics
    plt.axvline(mean_val, color="r", linestyle="--", label=f"Mean ({mean_val:.2f} dB)")
    plt.axvline(
        median_val, color="g", linestyle="--", label=f"Median ({median_val:.2f} dB)"
    )
    plt.axvline(p5, color="orange", linestyle=":", label=f"5th percentile")
    plt.axvline(p95, color="orange", linestyle=":", label=f"95th percentile")

    # Add labels and title
    plt.xlabel("Log-Intensity (dB)")
    plt.ylabel("Frequency")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Return statistics dictionary for potential further use
    return {
        "mean": mean_val,
        "median": median_val,
        "p1": p1,
        "p5": p5,
        "p10": p10,
        "p90": p90,
        "p95": p95,
        "p99": p99,
    }

In [ ]:
# Compare histograms of different patch types
print(
    "Histograms and statistics of the symmetrized and preserved patches are commented out to save time, they are all similar to the original data."
)
if "results" in locals():
    print("\n=== Original Image Histogram ===\n")
    orig_stats = plot_intensity_histogram(
        results["original_data"], title="Original SAR Image"
    )

    # print("\n=== Original Patches Histogram ===\n")
    # orig_patches_stats = plot_intensity_histogram(
    #     results["original_patches"], title="Original Patches"
    # )

    # print("\n=== Symmetrized Patches Histogram ===\n")
    # sym_patches_stats = plot_intensity_histogram(
    #     results["symmetrized_patches"], title="Symmetrized Patches"
    # )

    # print("\n=== Preserved Patches Histogram ===\n")
    # # Note: preserved_patches are already squared in the pipeline
    # pres_patches_stats = plot_intensity_histogram(
    #     results["patches"], title="Preserved Patches", is_squared=True
    # )

    # Print comparison summary
    print("\n=== Summary Comparison (Median values) ===\n")
    # print(f"Original Image: {orig_stats['median']:.2f} dB")
    # print(f"Original Patches: {orig_patches_stats['median']:.2f} dB")
    # print(f"Symmetrized Patches: {sym_patches_stats['median']:.2f} dB")
    # print(f"Preserved Patches: {pres_patches_stats['median']:.2f} dB")
else:
    print("No processed results available yet. Run the preprocessing cells first.")

In [ ]:
# Visualize the results
visualize_1_patch_processing_and_statistics(
    results["original_patches"][0],  # First original patch
    results["symmetrized_patches"][0],  # First symmetrized patch
    results["patches"][0],  # First preserved patch
    results["scatterer_masks"][0],  # First scatterer mask
)

In [ ]:
# Test writing a single file to HDF5
test_output = output_dir / "test_sample.h5"
write_success = write_hdf5(results["patches"], test_output)
if not write_success:
    print("Failed to write HDF5 file!")

In [ ]:
# Verify the HDF5 file was created and is readable
def verify_hdf5(file_path):
    """Verify HDF5 file can be read and show summary."""
    if not os.path.exists(file_path):
        print(f"File {file_path} does not exist!")
        return False

    with h5py.File(file_path, "r") as f:
        print(f"HDF5 file: {file_path}")
        print(f"Keys: {list(f.keys())}")
        print(f"Attributes: {dict(f.attrs)}")
        print("Dataset shapes:")
        for key in f.keys():
            print(f"  {key}: {f[key].shape}")

        # Show a sample patch
        plt.figure(figsize=(12, 4))

        plt.subplot(1, 3, 1)
        patch = f["patches"][0]
        plt.imshow(patch[:, :, 0], cmap="gray")
        plt.title("Real Part")
        plt.colorbar()

        plt.subplot(1, 3, 2)
        plt.imshow(patch[:, :, 1], cmap="gray")
        plt.title("Imaginary Part")
        plt.colorbar()

        plt.subplot(1, 3, 3)
        plt.imshow(patch[:, :, 0] ** 2 + patch[:, :, 1] ** 2, cmap="gray")
        plt.title("Intensity")
        plt.colorbar()

        plt.tight_layout()
        plt.show()

        return True


verify_hdf5(test_output)